#### Brochure Generator

In [1]:
# Load Packages
import os
import sys
import json
import asyncio
import subprocess
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

NOTEBOOK_DIR = os.path.dirname(os.path.abspath(globals().get("__vsc_ipynb_file__", os.getcwd())))
SCRAPER_SCRIPT = os.path.join(NOTEBOOK_DIR, "scraper.py")

In [2]:
# Scraper

def _run_scraper_sync(url, mode):
    result = subprocess.run(
        [sys.executable, SCRAPER_SCRIPT, url, "--mode", mode],
        capture_output=True, text=True, encoding="utf-8", check=True,
    )
    return result.stdout

async def fetch_website_content(url):
    return await asyncio.to_thread(_run_scraper_sync, url, "content")

async def fetch_website_links(url):
    links = await asyncio.to_thread(_run_scraper_sync, url, "links")
    return [link for link in links.splitlines() if link]

In [3]:
# Prompting

# System Prompt

link_system_prompt = """
You are provided with a numbered list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
Respond in JSON referencing each chosen link by its index number - do not retype the URL yourself.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "index": 3},
        {"type": "careers page", "index": 7}
    ]
}
"""

#  User Prompt

async def get_user_prompt(url, links):
    user_prompt = f"""
Here is the numbered list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
and respond with their index numbers in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    user_prompt += "\n".join(f"{i}: {link}" for i, link in enumerate(links))
    return user_prompt

In [4]:
# Relevant Links

async def select_relevant_links(url):
    links = await fetch_website_links(url)
    openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    response = openai.chat.completions.create(
    model="llama3.2",
    messages=[
        {"role": "system", "content": link_system_prompt},
        {"role": "user", "content": await get_user_prompt(url, links)}
    ],
    response_format={"type": "json_object"}
    )

    result = response.choices[0].message.content
    selected = json.loads(result)

    resolved = []
    for item in selected["links"]:
        index = item.get("index")
        if isinstance(index, int) and 0 <= index < len(links):
            resolved.append({"type": item.get("type", ""), "url": links[index]})

    return {"links": resolved}
        

In [5]:
print(await select_relevant_links("https://www.accenture.com/sg-en"))

{'links': [{'type': 'careers page', 'url': 'https://www.accenture.com/sg-en/careers'}, {'type': 'about company index', 'url': 'https://www.accenture.com/sg-en/about/company-index'}, {'type': 'case studies talent organization', 'url': 'https://www.accenture.com/sg-en/case-studies/talent-organization/ymca-transforms-crisis-into-capability'}, {'type': 'insights industrial research-reinventing-human-ai-engineering', 'url': 'https://www.accenture.com/sg-en/insights/industrial/reinventing-human-ai-engineering'}, {'type': 'insights software-platforms-ai-agents-rewriting-platform-playbook', 'url': 'https://www.accenture.com/sg-en/insights/software-platforms/ai-agents-rewriting-platform-playbook'}]}


In [6]:
async def fetch_page_and_relevant_links(url):
    content = await fetch_website_content(url)
    relevant_links = await select_relevant_links(url)
    result = f"## Landing Page:\n\n{content}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += await fetch_website_content(link["url"])
    return result

In [7]:
print(await fetch_page_and_relevant_links("https://www.accenture.com/sg-en"))

## Landing Page:

Singapore | Let There Be Change | Accenture

Skip to main content
Skip to footer
Menu
Accenture
Accenture
Close Menu
What we do
Back
What we do
Capabilities
Capabilities
Artificial Intelligence and Data
Cloud
Customer Service
Cybersecurity
Digital Engineering and Manufacturing
Ecosystem Partners
Emerging Technology
Finance and Risk Management
Infrastructure and Capital Projects
Learning
Managed Services
Marketing and Experience
Metaverse
Sales and Commerce
Strategy
Supply Chain
Sustainability
Talent and Organization
Technology Transformation
Industries
Industries
Aerospace and Defense
Automotive
Banking
Capital Markets
Chemicals
Communications and Media
Consumer Goods and Services
Energy
Health
High Tech
Industrial
Insurance
Life Sciences
Natural Resources
Public Service
Private Equity
Retail
Software and Platforms
Travel
Utilities
What we think
Who we are
Back
About Accenture
Our organization
Our organization
Reinvention Services
Leaders
Locations
360° Value Report
C

In [8]:
# Brochure Prompt 

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [9]:
async def get_brochure_user_prompt(company_name, url):
    user_prompt = f""" 
    You are looking at a company called: {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """
    user_prompt += await fetch_page_and_relevant_links(url)
    user_prompt = user_prompt[:5000] # Truncate if more than 5,000 characters
    return user_prompt

In [10]:
print(await get_brochure_user_prompt("Accenture", "https://www.accenture.com/sg-en"))

 
    You are looking at a company called: Accenture
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.


    ## Landing Page:

Singapore | Let There Be Change | Accenture

Skip to main content
Skip to footer
Menu
Accenture
Accenture
Close Menu
What we do
Back
What we do
Capabilities
Capabilities
Artificial Intelligence and Data
Cloud
Customer Service
Cybersecurity
Digital Engineering and Manufacturing
Ecosystem Partners
Emerging Technology
Finance and Risk Management
Infrastructure and Capital Projects
Learning
Managed Services
Marketing and Experience
Metaverse
Sales and Commerce
Strategy
Supply Chain
Sustainability
Talent and Organization
Technology Transformation
Industries
Industries
Aerospace and Defense
Automotive
Banking
Capital Markets
Chemicals
Communications and Media
Consumer Goods and Services
Energy
Health
High Tech
Industrial
Insurance
Life Sciences
Nat

In [11]:
async def create_brochure(company_name, url):
    openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    response = openai.chat.completions.create(
        model="llama3.2",
        messages = [
            {"role": "system", "content" : brochure_system_prompt},
            {"role": "user", "content": await get_brochure_user_prompt(company_name, url)}
        ]
        )
    
    result = response.choices[0].message.content
    display(Markdown(result))

In [12]:
await create_brochure ("Accenture", "https://www.accenture.com/sg-en")

# Accenture Brochure

## About Us

Accenture is a global leadership consulting firm that drives transformation for clients across all industries. With over 450,000 employees worldwide, we are one of the largest employers in the world.

## Mission and Values

At Accenture, our mission is to become the world's most inclusive and innovative consulting firm. We value:

* Embracing difference
* Empowering clients
* Delivering high-quality results
* Fostering a culture of innovation and learning
* Prioritizing sustainability and social responsibility

## Our Work

We help clients across all industries to achieve their digital transformation goals, navigate complex market trends, and drive growth in a rapidly changing world. Our expertise spans consulting, technology, and outsourcing.

## Customers

Some of our notable clients include:

* Banking and financial services
* Healthcare and life sciences
* Insurance and reinsurance
* IT-enabled services
* Manufacturing and process industries
* Telecommunications and media

## Careers

We are committed to creating a culture of diversity, equity, and inclusion in the workplace. We offer a wide range of career opportunities across various disciplines, including:

* Management consulting
* Technology and engineering
* Digital and innovation
* Health services
* Financial and banking

Our employees come from diverse backgrounds and cultures, and we strive to be an employer of choice for talented professionals worldwide.

## Locations

We have offices in over 120 countries, making us a truly global company.

[Accompanying image: A map of Accenture's global presence]

## Join Us

If you are passionate about delivering high-quality results, embracing change, and working with diverse teams of talent from around the world, then we want to hear from you.

Apply online now at [Accenture Careers page]

Or follow us on social media:

[LinkedIn, Twitter, Facebook, Instagram, YouTube]

In [13]:
async def stream_brochure(company_name, url):
    openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    stream = openai.chat.completions.create(
        model="llama3.2",
        messages = [
            {"role": "system", "content" : brochure_system_prompt},
            {"role": "user", "content": await get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
        )
    
    response="" # initalize as empty
    display_handle = display(Markdown(""), display_id=True) # initalize display
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
    

In [14]:
await stream_brochure("Accenture", "https://www.accenture.com/sg-en")

# Accenture Brochure

## About Us

Accenture is a global management consulting and professional services firm. With a rich history dating back to 1969, we have established ourselves as a trusted partner for clients around the world.

## Company Culture

At Accenture, we value integrity, diversity, and inclusion in our workplace culture. Our employees thrive in an environment that encourages collaboration, innovation, and learning. We foster a culture of professional growth and development, providing opportunities for individuals to grow and progress in their careers.

## Customers

Accenture works with numerous clients across various industries, including healthcare, energy, finance, and retail. Our professionals deliver solutions that drive business outcomes, improve operational efficiency, and enhance customer experiences.

## Key Industries

* Healthcare
* Energy
* Finance
* Retail
* Technology

## Careers & Jobs

Are you looking to join a company that values collaboration, innovation, and professional growth? Accenture offers a wide range of career opportunities in consulting, technology, outsourcing, and digital services. As an employee of Accenture, you'll have access to:

* Professional development training and certification programs
* Collaborative and diverse work environment
* Competitive compensation and benefits package

Apply for current job openings on our [Careers](link) page.

## Locations

Accenture operates in over 120 offices across more than 50 countries. Find your nearest office location by visiting our [Locations](link) page.

## Community Engagement

At Accenture, we're committed to making a positive impact on our communities and the world at large. We strive to contribute to causes that support education, disaster relief, and environmental sustainability.

# Investing in Accenture

For investors seeking a trusted partner for growth and innovation, Accenture offers:

* A strong track record of financial performance and stability
* Diversified revenue streams across various industries and services
* A committed investment in research and development to drive future growth

Invest in our share offer or learn more about our [Financial Highlights](link) page.

# Get in Touch

Stay connected with Accenture on social media:

 Linkedin: (link)
 Twitter: (link)
 Facebook: (link)
 Instagram: (link)
 YouTube: (link)

Alternatively, you can contact us via our website and get in touch with the latest news, updates, and stories from around the world.

## 2026 Accenture. All Rights Reserved.